# Lesson 07 - Planning Design Pattern

Ipinapakita ng notebook na ito ang **Planning Design Pattern** para sa mga AI agent gamit ang Microsoft Agent Framework.
Matututuhan mong paano hatiin ang mga kumplikadong kahilingan sa paglalakbay sa mga nakaayos na subtasks, itatalaga ang mga ito sa mga espesyalistang agent,
at isakatuparan ang resulta ng plano — lahat gamit ang nakaayos na output na pinapagana ng mga Pydantic na modelo.


## Setup


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Pagpira-piraso ng Gawain

Ang pagpira-piraso ng gawain ang puso ng disenyo ng planning pattern. Sa halip na hingin sa isang agent na hawakan ang isang komplikadong kahilingan mula simula hanggang dulo, hinahati namin ang problema sa mas maliliit, malinaw na **mga subtask**. Bawat subtask ay itinalaga sa isang espesyalistang agent (halimbawa, flights, hotels, activities) na may malinaw na mga prayoridad at pagkakasunod-sunod ng mga dependency.

Ang paraang ito ay nagbibigay ng ilang benepisyo:
- **Linaw**: bawat subtask ay may iisang responsibilidad.
- **Pagsabay-sabay**: mga independiyenteng subtask ay maaaring patakbuhin nang sabay-sabay.
- **Katiyakan**: ang mga pagkabigo ay naisasentro lamang sa mga indibidwal na subtask.
- **Pagsubaybay sa badyet**: ang mga gastos ay tinataya kada subtask at pinagsasama-sama.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Paglikha ng Planning Agent na may Istrakturadong Output

Ang planning agent ay kumikilos bilang isang **front desk coordinator**. Kapag binigyan ng isang mataas na antas na kahilingan sa paglalakbay, gumagawa ito ng isang istrakturadong `TravelPlan` — hinahati-hati ang kahilingan sa mga subtask, nagtatakda ng mga prayoridad, at tinutukoy ang mga dependencies upang ang isang concierge o execution layer ay makagawa ng gawain.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Pagsasakatuparan ng Plano gamit ang mga Espesyalistang Kasangkapan

Kapag nakabuo na ang front desk agent ng isang maayos na plano, isinasagawa ito ng **concierge agent**.
Bawat espesyalistang kasangkapan ay humahawak ng isang kategorya ng subtask (mga flight, hotel, aktibidad). Ang concierge
ay inuulit ang mga subtask ng plano ayon sa pagkakasunud-sunod ng kaginhawaan at ipinapadala ang bawat isa sa
karampatang kasangkapan.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Buod

Sa araling ito natutunan mo ang **Planning Design Pattern** para sa mga AI agent:

1. **Task Decomposition** — Ang isang front desk planning agent ay hinahati ang isang kumplikadong kahilingan sa paglalakbay sa mga
   istrukturadong subtasks gamit ang mga modelong Pydantic, na nag-a-assign ng bawat isa sa isang espesyalistang agent na may mga prayoridad
   at dependencies.
2. **Structured Output** — Sa pamamagitan ng pagpapasa ng `response_format` ang agent ay nagbabalik ng isang validated na
   `TravelPlan` object sa halip na malayang anyo ng teksto, na nagpapasiguro ng maaasahang pagpoproseso sa susunod.
3. **Plan Execution** — Ang isang concierge agent ay inuulit ang mga subtasks gamit ang mga espesyalistang tool
   (`book_flight`, `reserve_hotel`, `book_activity`) upang maisagawa ang plano at iulat ang mga resulta.

Pinaghihiwalay ng pattern na ito ang *kung ano ang gagawin* (planning) mula sa *kung paano ito gagawin* (execution), na nagpapadali sa mga agent
na maging modular, masusubukan, at mas madaling palawakin.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Paunawa**:
Ang dokumentong ito ay isinalin gamit ang AI translation service na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagamat nagsusumikap kaming maging tumpak, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o di-katumpakan. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasaling tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na maaaring magmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
